<a href="https://colab.research.google.com/github/Prasadn1234/DataScience/blob/main/Week_6_Assignment_13062026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Healthcare Multi-Role Clinical Intelligence System
## Capstone Assignment - 8-Week Progressive Exercise

**Course**: Agentic AI Bootcamp - Staff-Level System Design

---

## Assignment Overview

In this capstone assignment, you will build a **production-grade multi-agent healthcare system** that synthesizes patient data, clinical test results, and imaging reports across multiple healthcare roles.

### What You Will Build

| Component | Description |
|-----------|-------------|
| **Lab Value Interpreter** | Analyze CBC, liver, kidney, thyroid panels |
| **Radiology Analysis Agent** | Interpret X-Ray, CT, MRI, Ultrasound reports |
| **Allergy Cross-Reactivity Checker** | Identify dangerous allergen relationships |
| **Clinical Decision Support** | Multi-agent orchestration with role-based access |
| **Medical Knowledge Graph** | Graph-based clinical reasoning |

### Dataset Overview

| Category | Files | Description |
|----------|-------|-------------|
| **Patient Records** | Asthma samples, Discharge summaries | Patient history |
| **Allergy Records** | 17 types (Food, Medication, Environment, Skin) | Allergen data |
| **Clinician/Doctor** | CBC, Liver, Kidney, Thyroid, Heart, BMI reports | Lab results |
| **Radiologist** | X-Ray, CT, MRI, Ultrasound reports | Imaging studies |

### 8-Week Structure

| Week | Module | Difficulty |
|------|--------|------------|
| 1 | Environment & Data Loading | Beginner |
| 2 | Document Processing & Entity Extraction | Beginner |
| 3 | Vector Store & Embeddings | Intermediate |
| 4 | Single Agent Design | Intermediate |
| 5 | Multi-Agent Orchestration | Advanced |
| 6 | Medical Knowledge Graph | Advanced |
| 7 | Advanced Observability | Expert |
| 8 | Production Deployment | Expert |

### Grading Rubric

- **Functionality (40%)**: All components work correctly
- **Code Quality (20%)**: Clean, documented, follows best practices
- **Observability (20%)**: Proper Langfuse tracing throughout
- **Healthcare Domain (20%)**: Correct medical logic and safety checks

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


---
# WEEK 1: Environment Setup & Data Loading
---

## Learning Objectives
- Configure development environment in Google Colab
- Initialize Langfuse observability from the start
- Load and explore clinical documents

## Difficulty: Beginner

**HINT**: Environment setup is foundational. Pay attention to:
- Colab vs local environment detection
- Secure API key handling
- Data directory path configuration

In [ ]:
# ============================================================================
# WEEK 1: ENVIRONMENT SETUP (PROVIDED - Run this cell as-is)
# ============================================================================

import subprocess, sys

_packages = ["openai==1.59.6", "langfuse==2.57.1", "langchain==0.3.14", "langchain-openai==0.2.14", "langchain-community==0.3.14", "langchain-core==0.3.29", "chromadb", "networkx", "pyvis", "python-docx", "openpyxl", "pdfplumber", "plotly", "seaborn", "pydantic>=2.0", "tenacity", "rich", "gradio", "python-dotenv", "numpy", "pandas", "matplotlib"]

print("Installing packages (this may take 1-2 minutes)...")
result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q"] + _packages,
    capture_output=True, text=True
)

if result.returncode != 0:
    for line in result.stderr.split("\n"):
        if line.strip() and "dependency resolver" not in line.lower() and "notice" not in line.lower():
            print(line)

# ---------- Verify EVERY import ----------
_verify = [
    ("openai", "openai"),
    ("langfuse", "langfuse"),
    ("langchain", "langchain"),
    ("langchain_openai", "langchain_openai"),
    ("chromadb", "chromadb"),
    ("networkx", "networkx"),
    ("pyvis", "pyvis.network"),
    ("docx", "docx"),
    ("pdfplumber", "pdfplumber"),
    ("plotly", "plotly"),
    ("seaborn", "seaborn"),
    ("pydantic", "pydantic"),
    ("tenacity", "tenacity"),
    ("rich", "rich"),
    ("gradio", "gradio"),
    ("dotenv", "dotenv"),
    ("fastapi", "fastapi"),
    ("uvicorn", "uvicorn"),
]

_failed = []
for name, imp in _verify:
    try:
        __import__(imp)
    except ImportError:
        _failed.append(name)

if _failed:
    msg = f"FATAL: These packages failed to import: {', '.join(_failed)}\n"
    msg += "Try: Runtime > Restart runtime, then re-run this cell."
    raise ImportError(msg)

print("=" * 60)
print(f"All {len(_verify)} packages installed and verified!")
print("=" * 60)

In [ ]:
# ============================================================================
# IMPORTS AND ENVIRONMENT DETECTION (PROVIDED)
# ============================================================================

import os
import sys
import json
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime
from typing import List, Dict, Optional, Any, Tuple
from dataclasses import dataclass, field
from enum import Enum
import warnings
warnings.filterwarnings('ignore')
import subprocess

IN_COLAB = 'google.colab' in sys.modules

DATASET_REPO = "https://github.com/AI-Project-Lab/IK-pwc-agenticai-datasets.git"
DATASET_PROJECT = "healthcare"

if IN_COLAB:
    print("Running in Google Colab")

    # --- API Key Configuration ---
    try:
        from google.colab import userdata
        os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')
        os.environ['LANGFUSE_SECRET_KEY'] = userdata.get('LANGFUSE_SECRET_KEY')
        os.environ['LANGFUSE_PUBLIC_KEY'] = userdata.get('LANGFUSE_PUBLIC_KEY')
        _host = userdata.get('LANGFUSE_HOST') or ''
        os.environ['LANGFUSE_HOST'] = _host if _host.startswith('http') else 'https://cloud.langfuse.com'
        print("API keys loaded from Colab Secrets")
    except Exception as e:
        print(f"Colab Secrets not available: {e}")
        import getpass
        os.environ['OPENAI_API_KEY'] = getpass.getpass('Enter OpenAI API Key: ')
        os.environ['LANGFUSE_SECRET_KEY'] = getpass.getpass('Enter Langfuse Secret Key: ')
        os.environ['LANGFUSE_PUBLIC_KEY'] = getpass.getpass('Enter Langfuse Public Key: ')
        os.environ['LANGFUSE_HOST'] = 'https://cloud.langfuse.com'

    # --- Dataset Ingestion from GitHub ---
    dataset_path = "/content/datasets"
    if not os.path.exists(f"{dataset_path}/{DATASET_PROJECT}"):
        print(f"Cloning dataset from {DATASET_REPO}...")
        subprocess.run(["git", "clone", "--depth", "1", DATASET_REPO, dataset_path],
                       check=True, capture_output=True)
        print("Dataset cloned successfully!")
    else:
        print("Dataset already available.")
    DATA_DIR_BASE = f"{dataset_path}/{DATASET_PROJECT}"
else:
    print("Running in local environment")
    from dotenv import load_dotenv
    load_dotenv()
    print("API keys loaded from .env file")

    # --- Dataset Ingestion from GitHub ---
    datasets_parent = Path('.').resolve().parent.parent / 'datasets'
    if not (datasets_parent / DATASET_PROJECT).exists():
        print(f"Cloning dataset from {DATASET_REPO}...")
        subprocess.run(["git", "clone", "--depth", "1", DATASET_REPO, str(datasets_parent)],
                       check=True, capture_output=True)
        print("Dataset cloned successfully!")
    else:
        print("Dataset already available locally.")
    DATA_DIR_BASE = str(datasets_parent / DATASET_PROJECT)

# Validate API keys
required_keys = ['OPENAI_API_KEY', 'LANGFUSE_SECRET_KEY', 'LANGFUSE_PUBLIC_KEY']
missing_keys = [key for key in required_keys if not os.environ.get(key)]
if missing_keys:
    raise EnvironmentError(f"Missing required API keys: {missing_keys}")

DATA_DIR = Path(DATA_DIR_BASE)
print(f"\nAll API keys validated successfully")
print(f"Data directory: {DATA_DIR}")
print(f"Directory exists: {DATA_DIR.exists()}")
if DATA_DIR.exists():
    file_count = sum(1 for _ in DATA_DIR.rglob('*') if _.is_file())
    print(f"Total files found: {file_count}")

In [ ]:
# ============================================================================
# TODO 1.1: INITIALIZE LANGFUSE CLIENT
# ============================================================================

from langfuse import Langfuse
from langfuse.callback import CallbackHandler as LangfuseCallbackHandler
from openai import OpenAI

# Initialize Langfuse client
langfuse = Langfuse(
    secret_key=os.environ.get('LANGFUSE_SECRET_KEY'),
    public_key=os.environ.get('LANGFUSE_PUBLIC_KEY'),
    host=os.environ.get('LANGFUSE_HOST', 'https://cloud.langfuse.com')
)

# Verify connection
langfuse.auth_check()
print("Langfuse connection verified!")

# Create session ID
SESSION_ID = f"healthcare-master-{datetime.now().strftime('%Y%m%d-%H%M%S')}"
PROJECT_NAME = "healthcare-clinical-intelligence"

# Initialize OpenAI client
openai_client = OpenAI()

print(f"Session ID: {SESSION_ID}")
print(f"Project: {PROJECT_NAME}")
print(f"Dashboard: {os.environ.get('LANGFUSE_HOST', 'https://cloud.langfuse.com')}/project")

In [ ]:
# ============================================================================
# VALIDATION CELL 1.1: Verify Langfuse Setup
# ============================================================================

def validate_langfuse_setup():
    """Validate that Langfuse is properly configured."""
    errors = []

    if langfuse is None:
        errors.append("Langfuse client not initialized")
    else:
        try:
            langfuse.auth_check()
        except Exception as e:
            errors.append(f"Langfuse auth failed: {e}")

    if not SESSION_ID.startswith("healthcare-"):
        errors.append("SESSION_ID should start with 'healthcare-'")

    if errors:
        print("VALIDATION FAILED:")
        for e in errors:
            print(f"  - {e}")
        return False

    print("VALIDATION PASSED: Langfuse setup correct!")
    return True

validate_langfuse_setup()

In [ ]:
# ============================================================================
# TODO 1.2: CREATE TRACED HELPER FUNCTIONS
# ============================================================================

def traced_embedding(text: str, trace_name: str = "embedding") -> List[float]:
    """Generate embedding with full Langfuse tracing."""
    trace = langfuse.trace(
        name=trace_name,
        session_id=SESSION_ID,
        metadata={"text_length": len(text)}
    )
    generation = trace.generation(
        name="openai-embedding",
        model="text-embedding-3-small",
        input=text[:500]
    )
    response = openai_client.embeddings.create(
        model="text-embedding-3-small",
        input=text
    )
    embedding = response.data[0].embedding
    generation.end(
        output={"dimensions": len(embedding)},
        usage={"total_tokens": response.usage.total_tokens}
    )
    return embedding


def traced_completion(prompt: str, system: str = "", trace_name: str = "completion",
                      model: str = "gpt-4o-mini", temperature: float = 0) -> str:
    """Generate completion with full Langfuse tracing."""
    trace = langfuse.trace(name=trace_name, session_id=SESSION_ID)
    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": prompt})

    generation = trace.generation(
        name="openai-completion",
        model=model,
        input=messages
    )
    response = openai_client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=temperature
    )
    result = response.choices[0].message.content
    generation.end(
        output=result,
        usage={
            "prompt_tokens": response.usage.prompt_tokens,
            "completion_tokens": response.usage.completion_tokens,
            "total_tokens": response.usage.total_tokens
        }
    )
    return result


def get_langfuse_handler(trace_name: str, tags: List[str] = None) -> LangfuseCallbackHandler:
    """Get a LangChain callback handler for automatic tracing."""
    return LangfuseCallbackHandler(
        secret_key=os.environ.get('LANGFUSE_SECRET_KEY'),
        public_key=os.environ.get('LANGFUSE_PUBLIC_KEY'),
        host=os.environ.get('LANGFUSE_HOST', 'https://cloud.langfuse.com'),
        session_id=SESSION_ID,
        trace_name=trace_name,
        tags=tags or []
    )


print("Traced helper functions defined")

In [ ]:
# ============================================================================
# TODO 1.3: DISCOVER CLINICAL DATA FILES
# ============================================================================

def discover_clinical_data(data_dir: Path) -> Dict[str, List[Path]]:
    """Discover all clinical documents organized by category."""
    categories = {
        'patient': [],
        'clinician': [],
        'radiologist': [],
        'training': [],
        'allergy_food': [],
        'allergy_medication': [],
        'allergy_environment': [],
        'allergy_skin': []
    }

    if not data_dir.exists():
        print(f"Warning: Data directory not found at {data_dir}")
        return categories

    supported = {'.docx', '.pdf', '.xlsx'}
    food_allergens = {'peanut', 'wheat', 'milk', 'egg', 'soy', 'fish', 'shellfish', 'gluten', 'food'}
    med_allergens  = {'ibuprofen', 'penicillin', 'aspirin', 'sulfa', 'nsaid', 'medication', 'drug'}
    env_allergens  = {'dust', 'pollen', 'mold', 'dander', 'latex', 'bee', 'environment'}
    skin_allergens = {'skin', 'contact', 'dermat', 'eczema', 'urticaria'}

    for item in data_dir.rglob('*'):
        if not item.is_file() or item.suffix.lower() not in supported:
            continue
        path_str = str(item).lower()

        if 'patient' in path_str or 'asthma' in path_str or 'discharge' in path_str:
            categories['patient'].append(item)
        elif 'radiolog' in path_str or 'xray' in path_str or 'x-ray' in path_str \
                or 'ct_' in path_str or 'mri' in path_str or 'ultrasound' in path_str:
            categories['radiologist'].append(item)
        elif 'training' in path_str or 'glossary' in path_str or 'reference' in path_str:
            categories['training'].append(item)
        elif 'allerg' in path_str:
            if any(k in path_str for k in food_allergens):
                categories['allergy_food'].append(item)
            elif any(k in path_str for k in med_allergens):
                categories['allergy_medication'].append(item)
            elif any(k in path_str for k in env_allergens):
                categories['allergy_environment'].append(item)
            elif any(k in path_str for k in skin_allergens):
                categories['allergy_skin'].append(item)
            else:
                categories['allergy_food'].append(item)  # default allergy bucket
        elif any(k in path_str for k in ('cbc', 'liver', 'kidney', 'thyroid',
                                          'heart', 'bmi', 'clinician', 'doctor', 'lab')):
            categories['clinician'].append(item)

    return categories


# Discover data
if langfuse:
    trace = langfuse.trace(
        name="data-discovery",
        session_id=SESSION_ID,
        metadata={"data_dir": str(DATA_DIR)}
    )

clinical_files = discover_clinical_data(DATA_DIR)

print("\n" + "="*60)
print("HEALTHCARE DATASET DISCOVERY")
print("="*60)
total_files = 0
for category, files in clinical_files.items():
    print(f"\n{category.upper().replace('_', ' ')}: {len(files)} files")
    for f in files[:3]:
        print(f"  - {f.name}")
    if len(files) > 3:
        print(f"  ... and {len(files) - 3} more")
    total_files += len(files)

print(f"\nTotal clinical documents: {total_files}")

---
# WEEK 2: Document Processing & Entity Extraction
---

## Learning Objectives
- Parse heterogeneous clinical documents (DOCX, PDF, XLSX)
- Extract structured information from unstructured text
- Implement Named Entity Recognition for medical terms

## Difficulty: Beginner to Intermediate

**HEALTHCARE CHALLENGE**: Clinical documents have:
- Inconsistent formatting across departments
- Medical abbreviations and jargon
- Tables with lab values
- Mixed structured/unstructured content

In [ ]:
# ============================================================================
# TODO 2.1: IMPLEMENT CLINICAL DOCUMENT PROCESSOR
# ============================================================================

from docx import Document as DocxDocument
import pdfplumber
import openpyxl

class ClinicalDocumentProcessor:
    """Production-grade document processor for healthcare documents."""

    def __init__(self):
        self.supported_formats = ['.docx', '.pdf', '.xlsx']
        self.processed_docs = []

    def load_docx(self, file_path: Path) -> Dict[str, Any]:
        try:
            doc = DocxDocument(file_path)
            paragraphs = [p.text.strip() for p in doc.paragraphs if p.text.strip()]
            tables = []
            for table in doc.tables:
                rows = []
                for row in table.rows:
                    rows.append([cell.text.strip() for cell in row.cells])
                tables.append(rows)
            return {
                'filename': file_path.name,
                'format': 'docx',
                'paragraphs': paragraphs,
                'tables': tables,
                'full_text': '\n'.join(paragraphs),
                'table_text': self._tables_to_text(tables)
            }
        except Exception as e:
            print(f"Error loading {file_path.name}: {e}")
            return None

    def load_pdf(self, file_path: Path) -> Dict[str, Any]:
        try:
            text_content = []
            tables = []
            with pdfplumber.open(file_path) as pdf:
                for page in pdf.pages:
                    text = page.extract_text()
                    if text:
                        text_content.append(text)
                    page_tables = page.extract_tables()
                    if page_tables:
                        tables.extend(page_tables)
            return {
                'filename': file_path.name,
                'format': 'pdf',
                'paragraphs': text_content,
                'tables': tables,
                'full_text': '\n'.join(text_content),
                'table_text': self._tables_to_text(tables)
            }
        except Exception as e:
            print(f"Error loading PDF {file_path.name}: {e}")
            return None

    def load_xlsx(self, file_path: Path) -> Dict[str, Any]:
        try:
            wb = openpyxl.load_workbook(file_path, data_only=True)
            text_content = []
            tables = []
            for sheet in wb.worksheets:
                text_content.append(f"Sheet: {sheet.title}")
                rows = []
                for row in sheet.iter_rows(values_only=True):
                    row_data = [str(cell) if cell is not None else '' for cell in row]
                    if any(cell.strip() for cell in row_data):
                        rows.append(row_data)
                        text_content.append(' | '.join(row_data))
                if rows:
                    tables.append(rows)
            return {
                'filename': file_path.name,
                'format': 'xlsx',
                'paragraphs': text_content,
                'tables': tables,
                'full_text': '\n'.join(text_content),
                'table_text': self._tables_to_text(tables)
            }
        except Exception as e:
            print(f"Error loading Excel {file_path.name}: {e}")
            return None

    def _tables_to_text(self, tables: List) -> str:
        text_parts = []
        for i, table in enumerate(tables):
            text_parts.append(f"\nTable {i+1}:")
            for row in table:
                if isinstance(row, list):
                    text_parts.append(' | '.join(str(cell) for cell in row if cell))
        return '\n'.join(text_parts)

    def process_file(self, file_path: Path) -> Dict[str, Any]:
        suffix = file_path.suffix.lower()
        if suffix == '.docx':
            return self.load_docx(file_path)
        elif suffix == '.pdf':
            return self.load_pdf(file_path)
        elif suffix == '.xlsx':
            return self.load_xlsx(file_path)
        else:
            print(f"Unsupported format: {suffix}")
            return None

    def process_category_with_tracing(self, files: List[Path], category: str) -> List[Dict]:
        trace = langfuse.trace(
            name=f"doc-processing-{category}",
            session_id=SESSION_ID,
            metadata={"category": category, "file_count": len(files)}
        ) if langfuse else None

        processed = []
        for file_path in files:
            span = trace.span(name=f"process-{file_path.name}") if trace else None
            doc = self.process_file(file_path)
            if doc:
                doc['category'] = category
                doc['doc_id'] = f"{category.upper()}-{len(processed)+1:03d}"
                processed.append(doc)
            if span:
                span.end(output={"success": doc is not None})

        if trace:
            trace.update(output={"processed": len(processed)})
        return processed


doc_processor = ClinicalDocumentProcessor()
print("ClinicalDocumentProcessor initialized")

In [ ]:
# ============================================================================
# PROCESS ALL CLINICAL DOCUMENTS
# ============================================================================

# Process all categories
all_clinical_docs = {}
for category, files in clinical_files.items():
    if files:
        all_clinical_docs[category] = doc_processor.process_category_with_tracing(files, category)
        print(f"Processed {len(all_clinical_docs[category])} {category} documents")

total_processed = sum(len(docs) for docs in all_clinical_docs.values())
print(f"\nTotal documents processed: {total_processed}")

In [ ]:
# ============================================================================
# TODO 2.2: IMPLEMENT CLINICAL ENTITY EXTRACTOR
# ============================================================================

import re

class ClinicalEntityExtractor:
    """Extract clinical entities from medical documents."""

    def __init__(self):
        self.lab_patterns = {
            'hemoglobin': r'(?:Hemoglobin|Hgb|HGB)[:\s]+([\d.]+)\s*(g/dL|g/dl)?',
            'wbc':        r'(?:WBC|White Blood Cell)[:\s]+([\d,]+)\s*(/uL|/µL)?',
            'rbc':        r'(?:RBC|Red Blood Cell)[:\s]+([\d.]+)\s*(M/uL|million)?',
            'platelet':   r'(?:Platelet|PLT)[:\s]+([\d,]+)\s*(/uL|/µL)?',
            'glucose':    r'(?:Glucose|Blood Sugar)[:\s]+([\d.]+)\s*(mg/dL)?',
            'creatinine': r'(?:Creatinine|Creat)[:\s]+([\d.]+)\s*(mg/dL)?',
            'bilirubin':  r'(?:Bilirubin|Bili)[:\s]+([\d.]+)\s*(mg/dL)?',
            'ast':        r'(?:AST|SGOT)[:\s]+([\d.]+)\s*(U/L)?',
            'alt':        r'(?:ALT|SGPT)[:\s]+([\d.]+)\s*(U/L)?',
            'tsh':        r'(?:TSH)[:\s]+([\d.]+)\s*(mIU/L)?',
        }
        self.status_patterns = {
            'normal':   r'\b(NORMAL|Normal|WNL|Within Normal)\b',
            'high':     r'\b(HIGH|High|Elevated|Increased|Above)\b',
            'low':      r'\b(LOW|Low|Decreased|Below)\b',
            'critical': r'\b(CRITICAL|Critical|STAT|Urgent)\b'
        }
        self._allergen_list = [
            'peanut', 'wheat', 'milk', 'egg', 'soy', 'fish', 'shellfish', 'gluten',
            'penicillin', 'sulfa', 'aspirin', 'ibuprofen', 'nsaid', 'paracetamol',
            'dust', 'pollen', 'mold', 'pet dander', 'latex', 'bee'
        ]

    def extract_lab_values(self, text: str) -> List[Dict]:
        results = []
        for test, pattern in self.lab_patterns.items():
            for match in re.finditer(pattern, text, re.IGNORECASE):
                value = match.group(1).replace(',', '')
                unit  = match.group(2) if match.lastindex >= 2 else ''
                results.append({'test': test, 'value': value, 'unit': unit or ''})
        return results

    def extract_status(self, text: str) -> List[str]:
        statuses = []
        for status, pattern in self.status_patterns.items():
            if re.search(pattern, text):
                statuses.append(status)
        return statuses

    def extract_allergies(self, text: str) -> List[str]:
        text_lower = text.lower()
        return [a for a in self._allergen_list if a in text_lower]

    def process_document_with_tracing(self, doc: Dict) -> Dict:
        text = doc.get('full_text', '') + '\n' + doc.get('table_text', '')
        result = {
            'doc_id':     doc.get('doc_id'),
            'category':   doc.get('category'),
            'lab_values': self.extract_lab_values(text),
            'statuses':   self.extract_status(text),
            'allergies':  self.extract_allergies(text)
        }
        if langfuse:
            trace = langfuse.trace(
                name="entity-extraction",
                session_id=SESSION_ID,
                metadata={"doc_id": doc.get('doc_id')}
            )
            trace.update(output={
                "lab_count":     len(result['lab_values']),
                "status_count":  len(result['statuses']),
                "allergy_count": len(result['allergies'])
            })
        return result


entity_extractor = ClinicalEntityExtractor()
print("ClinicalEntityExtractor initialized")

In [ ]:
# ============================================================================
# VALIDATION CELL 2.1: Test Entity Extraction
# ============================================================================

def validate_entity_extraction():
    """Test the entity extractor with sample clinical text."""

    test_text = """
    CBC Report:
    Hemoglobin: 10.2 g/dL (LOW)
    WBC: 12,500 /uL (HIGH)
    Platelet: 180,000 /uL (NORMAL)

    Patient has documented peanut allergy.
    Avoid penicillin due to previous reaction.
    """

    lab_values = entity_extractor.extract_lab_values(test_text)
    statuses = entity_extractor.extract_status(test_text)
    allergies = entity_extractor.extract_allergies(test_text)

    print("Entity Extraction Test Results:")
    print(f"  Lab values found: {len(lab_values)}")
    for lv in lab_values:
        print(f"    - {lv}")
    print(f"  Statuses found: {statuses}")
    print(f"  Allergies found: {allergies}")

    # Validation checks
    passed = True
    if len(lab_values) < 1:
        print("  WARNING: Should find at least 1 lab value")
        passed = False
    if 'low' not in statuses and 'high' not in statuses:
        print("  WARNING: Should find LOW or HIGH status")
        passed = False
    if 'peanut' not in allergies:
        print("  WARNING: Should find peanut allergy")
        passed = False

    if passed:
        print("\nVALIDATION PASSED!")
    else:
        print("\nVALIDATION NEEDS WORK - implement the TODO sections above")

validate_entity_extraction()

In [ ]:
# ============================================================================
# WEEK 2.7: ENTITY EXTRACTION EXAMPLES - Lab Values
# ============================================================================

import re  # ensure re is in scope for extractor methods

print("ENTITY EXTRACTION EXAMPLES - Laboratory Values")
print("="*60)

lab_examples = [
    """CBC Report:
    Hemoglobin: 10.2 g/dL (LOW)
    WBC: 12,500 /uL (HIGH)
    Platelet: 180,000 /uL (NORMAL)""",

    """Liver Function Test:
    AST: 85 U/L (ELEVATED)
    ALT: 92 U/L (HIGH)
    Bilirubin: 1.8 mg/dL""",

    """Kidney Function:
    Creatinine: 2.1 mg/dL (HIGH)
    BUN: 28 mg/dL (ELEVATED)""",

    """Thyroid Panel:
    TSH: 8.5 mIU/L (HIGH)
    Free T4: 0.6 ng/dL (LOW)"""
]

for i, example in enumerate(lab_examples, 1):
    print(f"\n[Example {i}]")
    print("-" * 40)

    labs = entity_extractor.extract_lab_values(example)
    statuses = entity_extractor.extract_status(example)

    print(f"Extracted lab values:")
    for lab in labs:
        print(f"  - {lab['test']}: {lab['value']} {lab['unit']}")
    print(f"Status indicators: {statuses}")

---
# WEEK 3: ChromaDB Vector Store & Embeddings
---

## Learning Objectives
- Understand semantic embeddings for medical text
- Set up ChromaDB for vector storage with tracing
- Index clinical documents for RAG retrieval

## Difficulty: Intermediate

**HEALTHCARE CHALLENGE**: Medical embeddings must capture:
- Medical terminology relationships (e.g., "MI" = "myocardial infarction" = "heart attack")
- Clinical context (symptoms + findings + diagnoses)
- Semantic similarity for retrieval-augmented generation

In [ ]:
# ============================================================================
# TODO 3.1: IMPLEMENT TRACED CLINICAL VECTOR STORE
# ============================================================================

import chromadb

class TracedClinicalVectorStore:
    """ChromaDB-based vector store with full Langfuse tracing."""

    def __init__(self, persist_directory: str = "./chroma_healthcare_db"):
        self.client = chromadb.PersistentClient(path=persist_directory)
        self.collections = {}

    def create_collection(self, name: str) -> chromadb.Collection:
        collection = self.client.get_or_create_collection(
            name=name,
            metadata={"hnsw:space": "cosine"}
        )
        self.collections[name] = collection
        return collection

    def chunk_document(self, text: str, chunk_size: int = 500, overlap: int = 50) -> List[str]:
        words = text.split()
        if not words:
            return [text]
        chunks = []
        step = chunk_size - overlap
        for i in range(0, len(words), step):
            chunk = ' '.join(words[i:i + chunk_size])
            if chunk.strip():
                chunks.append(chunk)
        return chunks if chunks else [text]

    def add_documents_with_tracing(self, collection_name: str, documents: List[Dict]):
        trace = langfuse.trace(
            name=f"indexing-{collection_name}",
            session_id=SESSION_ID
        ) if langfuse else None

        collection = self.create_collection(collection_name)

        all_ids, all_texts, all_embeddings, all_metadata = [], [], [], []

        for doc in documents:
            text = doc.get('full_text', '') + '\n' + doc.get('table_text', '')
            chunks = self.chunk_document(text)

            for i, chunk in enumerate(chunks):
                chunk_id = f"{doc['doc_id']}_chunk_{i}"
                embedding = traced_embedding(chunk, f"embed-{chunk_id}")
                all_ids.append(chunk_id)
                all_texts.append(chunk)
                all_embeddings.append(embedding)
                all_metadata.append({
                    'doc_id':      doc['doc_id'],
                    'filename':    doc['filename'],
                    'category':    doc['category'],
                    'chunk_index': i
                })

        if all_ids:
            collection.add(
                ids=all_ids,
                embeddings=all_embeddings,
                documents=all_texts,
                metadatas=all_metadata
            )

        if trace:
            trace.update(output={"chunks_indexed": len(all_ids)})
        print(f"Indexed {len(all_ids)} chunks into '{collection_name}'")

    def search_with_tracing(self, collection_name: str, query: str, n_results: int = 5) -> Dict:
        trace = langfuse.trace(
            name=f"search-{collection_name}",
            session_id=SESSION_ID,
            input={"query": query}
        ) if langfuse else None

        if collection_name not in self.collections:
            self.create_collection(collection_name)
        collection = self.collections[collection_name]

        query_embedding = traced_embedding(query, f"search-query-{collection_name}")

        # Guard: don't query an empty collection
        if collection.count() == 0:
            return {'documents': [[]], 'metadatas': [[]], 'distances': [[]]}

        results = collection.query(
            query_embeddings=[query_embedding],
            n_results=min(n_results, collection.count()),
            include=['documents', 'metadatas', 'distances']
        )

        if trace:
            trace.update(output={"result_count": len(results['documents'][0])})
        return results


vector_store = TracedClinicalVectorStore()
print("TracedClinicalVectorStore initialized")

In [ ]:
# ============================================================================
# INDEX CLINICAL DOCUMENTS
# ============================================================================

print("Indexing clinical documents into ChromaDB...")
print("(Each embedding operation will be traced to Langfuse)\n")

# Index each category
for category, docs in all_clinical_docs.items():
    if docs:
        vector_store.add_documents_with_tracing(f"healthcare_{category}", docs)

# Create unified collection
all_docs = []
for docs in all_clinical_docs.values():
    all_docs.extend(docs)

if all_docs:
    vector_store.add_documents_with_tracing("healthcare_all", all_docs)

print("\nIndexing complete!")

In [ ]:
# ============================================================================
# TODO 3.2: TEST SEMANTIC SEARCH
# ============================================================================
#
# Your task: Test the vector store with clinical queries
#
# Run searches for these healthcare queries and verify results make sense:
# ============================================================================

test_queries = [
    "What are the normal hemoglobin levels?",
    "Patient with peanut allergy symptoms",
    "Chest X-ray findings for pneumonia",
    "Liver function test interpretation",
    "Medication allergy to ibuprofen"
]

print("SEMANTIC SEARCH TEST")
print("="*60)

for i, query in enumerate(test_queries, 1):
    print(f"\n[Query {i}] '{query}'")
    print("-" * 50)

    results = vector_store.search_with_tracing("healthcare_all", query, n_results=2)

    if results['documents'] and results['documents'][0]:
        for j, (doc, meta, dist) in enumerate(zip(
            results['documents'][0],
            results['metadatas'][0],
            results['distances'][0]
        )):
            similarity = 1 - dist if dist else 0
            print(f"  Result {j+1} (Similarity: {similarity:.3f})")
            print(f"  Source: {meta.get('filename', 'N/A')} ({meta.get('category', 'N/A')})")
            print(f"  Preview: {doc[:100]}..." if doc else "  No content")
    else:
        print("  No results (implement search_with_tracing first)")

---
# WEEK 4: Single Agent Design
---

## Learning Objectives
- Design specialized clinical agents with structured outputs
- Implement Pydantic models for type-safe responses
- Create Lab Interpreter and Radiology Analyzer agents

## Difficulty: Intermediate to Advanced

**HEALTHCARE CHALLENGE - Clinical Decision Support**:
- Agents must provide evidence-based recommendations
- Outputs must include confidence scores
- Urgency levels are critical for patient safety
- All decisions must be traceable for audit

In [ ]:
# ============================================================================
# TODO 4.1: DEFINE PYDANTIC MODELS FOR STRUCTURED OUTPUT
# ============================================================================
#
# Your task: Create Pydantic models for agent responses
#
# HEALTHCARE REQUIREMENT: All clinical outputs MUST include:
# - Confidence score (0.0 to 1.0)
# - Urgency level (routine, urgent, stat, emergency)
# - Recommended actions
#
# These ensure patient safety and clinical accountability
# ============================================================================

from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import PydanticOutputParser


class LabInterpretation(BaseModel):
    """
    Structured output for lab result interpretation.

    TODO: Complete the field definitions with proper descriptions
    """
    test_name: str = Field(description="Name of the lab test")
    result_value: str = Field(description="The actual result value with units")
    status: str = Field(description="Status: NORMAL, LOW, HIGH, CRITICAL_LOW, CRITICAL_HIGH")
    clinical_significance: str = Field(description="What this result means clinically")
    potential_causes: List[str] = Field(description="Possible causes for abnormal results")
    recommended_actions: List[str] = Field(description="Suggested follow-up actions")
    urgency: str = Field(description="Urgency: routine, urgent, stat, emergency")
    confidence: float = Field(ge=0.0, le=1.0, description="Interpretation confidence")


class RadiologyFinding(BaseModel):
    """
    Structured output for radiology interpretation.

    TODO: Add fields for:
    - study_type: Type of imaging (X-Ray, CT, MRI, Ultrasound)
    - body_region: Body region examined
    - findings: List of key findings
    - impression: Overall impression/diagnosis
    - differential_diagnoses: List of possible diagnoses
    - recommendations: Follow-up recommendations
    - urgency: Finding urgency
    - confidence: Interpretation confidence
    """
    # YOUR CODE HERE
    study_type: str = Field(description="Type of imaging study")
    body_region: str = Field(description="Body region examined")
    findings: List[str] = Field(description="Key findings from the study")
    impression: str = Field(description="Overall impression/diagnosis")
    # TODO: Add remaining fields
    confidence: float = Field(ge=0.0, le=1.0)


class AllergyAssessment(BaseModel):
    """
    Structured output for allergy safety assessment.

    CRITICAL HEALTHCARE FIELDS:
    - cross_reactivities: Other allergens to avoid (e.g., penicillin -> cephalosporins)
    - emergency_protocol: What to do if exposure occurs

    TODO: Complete the model
    """
    allergen: str = Field(description="The allergen identified")
    category: str = Field(description="Category: Food, Medication, Environmental, Skin")
    severity: str = Field(description="Severity: MILD, MODERATE, SEVERE, CRITICAL")
    reactions: List[str] = Field(description="Known reactions to this allergen")
    cross_reactivities: List[str] = Field(description="Related allergens to avoid")
    safe_alternatives: List[str] = Field(description="Safe alternatives if applicable")
    emergency_protocol: str = Field(description="Emergency response if exposure occurs")
    confidence: float = Field(ge=0.0, le=1.0)


print("Pydantic models defined for structured agent outputs")

In [ ]:
# ============================================================================
# TODO 4.2: IMPLEMENT LAB INTERPRETER AGENT
# ============================================================================

class TracedLabInterpreterAgent:
    """Lab Interpreter Agent with full Langfuse tracing."""

    def __init__(self, model: str = "gpt-4o-mini"):
        self.model = model
        self.parser = PydanticOutputParser(pydantic_object=LabInterpretation)
        self.prompt = ChatPromptTemplate.from_messages([
            ("system", """You are an expert clinical laboratory medicine specialist.
Interpret lab results and provide clinical significance, potential causes, and recommendations.

Reference ranges:
- Hemoglobin: Male 13.5-17.5 g/dL, Female 12.0-16.0 g/dL
- WBC: 4,500-11,000 /uL
- Platelet: 150,000-400,000 /uL
- Glucose (fasting): 70-100 mg/dL
- Creatinine: 0.7-1.3 mg/dL
- AST: 10-40 U/L, ALT: 7-56 U/L
- TSH: 0.4-4.0 mIU/L

{format_instructions}"""),
            ("human", "Interpret this lab report:\n\n{lab_report}")
        ])

    def interpret(self, lab_report: str, patient_id: str = "anonymous") -> Dict:
        try:
            handler = get_langfuse_handler(f"lab-interpretation-{patient_id}", ["lab", "agent"])
            llm = ChatOpenAI(model=self.model, temperature=0, callbacks=[handler])
            chain = self.prompt | llm | self.parser
            result = chain.invoke({
                "lab_report": lab_report,
                "format_instructions": self.parser.get_format_instructions()
            })
            return {"success": True, "result": result.model_dump()}
        except Exception as e:
            return {"success": False, "error": str(e)}


lab_agent = TracedLabInterpreterAgent()
print("TracedLabInterpreterAgent initialized")

In [ ]:
# ============================================================================
# TODO 4.3: IMPLEMENT RADIOLOGY ANALYZER AGENT
# ============================================================================

class TracedRadiologyAgent:
    """Radiology Analyzer Agent with full Langfuse tracing."""

    def __init__(self, model: str = "gpt-4o-mini"):
        self.model = model
        self.parser = PydanticOutputParser(pydantic_object=RadiologyFinding)
        self.prompt = ChatPromptTemplate.from_messages([
            ("system", """You are an expert radiologist specializing in diagnostic imaging.
Analyze radiology reports and provide findings, differential diagnoses, and recommendations.
Flag urgent findings requiring immediate attention.

{format_instructions}"""),
            ("human", "Analyze this radiology report:\n\n{radiology_report}")
        ])

    def analyze(self, radiology_report: str, patient_id: str = "anonymous") -> Dict:
        try:
            handler = get_langfuse_handler(f"radiology-{patient_id}", ["radiology", "agent"])
            llm = ChatOpenAI(model=self.model, temperature=0, callbacks=[handler])
            chain = self.prompt | llm | self.parser
            result = chain.invoke({
                "radiology_report": radiology_report,
                "format_instructions": self.parser.get_format_instructions()
            })
            return {"success": True, "result": result.model_dump()}
        except Exception as e:
            return {"success": False, "error": str(e)}


radiology_agent = TracedRadiologyAgent()
print("TracedRadiologyAgent initialized")

In [ ]:
# ============================================================================
# TODO 4.4: IMPLEMENT ALLERGY SAFETY AGENT
# ============================================================================

class TracedAllergyAgent:
    """Allergy Safety Agent with full Langfuse tracing. SAFETY-CRITICAL."""

    def __init__(self, model: str = "gpt-4o-mini"):
        self.model = model
        self.parser = PydanticOutputParser(pydantic_object=AllergyAssessment)
        self.prompt = ChatPromptTemplate.from_messages([
            ("system", """You are an allergy and immunology specialist.
Assess allergy severity, identify cross-reactive allergens, and provide safety recommendations.
Patient safety is the top priority - always err on the side of caution.

Known cross-reactivities:
- Penicillin: amoxicillin, ampicillin, ~10% cross-react with cephalosporins
- Peanut: tree nuts, legumes (soybeans, lentils), lupin flour
- Latex: banana, avocado, kiwi, chestnut (latex-fruit syndrome)
- Aspirin: all NSAIDs including ibuprofen, naproxen
- Shellfish: all crustaceans (shrimp, crab, lobster)

{format_instructions}"""),
            ("human", "Assess this allergy documentation:\n\n{allergy_doc}")
        ])

    def assess(self, allergy_doc: str, patient_id: str = "anonymous") -> Dict:
        try:
            handler = get_langfuse_handler(f"allergy-{patient_id}", ["allergy", "agent", "safety-critical"])
            llm = ChatOpenAI(model=self.model, temperature=0, callbacks=[handler])
            chain = self.prompt | llm | self.parser
            result = chain.invoke({
                "allergy_doc": allergy_doc,
                "format_instructions": self.parser.get_format_instructions()
            })
            return {"success": True, "result": result.model_dump()}
        except Exception as e:
            return {"success": False, "error": str(e)}


allergy_agent = TracedAllergyAgent()
print("TracedAllergyAgent initialized")

In [ ]:
# ============================================================================
# VALIDATION CELL 4.1: Test All Agents
# ============================================================================

print("AGENT VALIDATION TESTS")
print("="*60)

# Test Lab Agent
print("\n[Test 1] Lab Interpreter Agent")
lab_report = """CBC Report:
Hemoglobin: 10.2 g/dL (LOW)
WBC: 12,500 /uL (HIGH)
Platelet: 180,000 /uL (NORMAL)"""

result = lab_agent.interpret(lab_report, "test-patient-001")
print(f"  Success: {result.get('success', False)}")
if result.get('success'):
    print(f"  Test: {result['result']['test_name']}")
    print(f"  Status: {result['result']['status']}")
    print(f"  Urgency: {result['result']['urgency']}")

# Test Radiology Agent
print("\n[Test 2] Radiology Analyzer Agent")
radiology_report = """Chest X-Ray:
Bilateral infiltrates in lower lobes.
No pleural effusion. Heart size normal.
Impression: Findings consistent with pneumonia."""

result = radiology_agent.analyze(radiology_report, "test-patient-002")
print(f"  Success: {result.get('success', False)}")

# Test Allergy Agent
print("\n[Test 3] Allergy Safety Agent")
allergy_doc = """Patient has documented peanut allergy.
Previous reaction: Severe anaphylaxis requiring epinephrine."""

result = allergy_agent.assess(allergy_doc, "test-patient-003")
print(f"  Success: {result.get('success', False)}")
if result.get('success'):
    print(f"  Cross-reactivities: {result['result']['cross_reactivities']}")

---
# WEEK 5: Multi-Agent Orchestration
---

## Learning Objectives
- Implement orchestration patterns with tracing
- Build role-based access control (RBAC)
- Handle inter-agent communication

## Difficulty: Advanced

**HEALTHCARE CHALLENGE - Role-Based Access**:
- Patients: Can only access their own allergy information
- Clinicians: Can access lab results and allergies
- Radiologists: Can only access imaging studies
- Coordinators: Full access for care coordination

In [ ]:
# ============================================================================
# TODO 5.1: IMPLEMENT CLINICAL ORCHESTRATOR
# ============================================================================

class ClinicalRole(Enum):
    PATIENT     = "patient"
    CLINICIAN   = "clinician"
    RADIOLOGIST = "radiologist"
    COORDINATOR = "coordinator"


class TracedClinicalOrchestrator:
    """Multi-agent orchestrator with full Langfuse tracing and RBAC."""

    def __init__(self):
        self.lab_agent      = lab_agent
        self.radiology_agent = radiology_agent
        self.allergy_agent  = allergy_agent
        self.vector_store   = vector_store

        self.role_permissions = {
            ClinicalRole.PATIENT:     ['allergy'],
            ClinicalRole.CLINICIAN:   ['lab', 'allergy'],
            ClinicalRole.RADIOLOGIST: ['radiology'],
            ClinicalRole.COORDINATOR: ['lab', 'radiology', 'allergy']
        }

    def route_query(self, query: str) -> str:
        q = query.lower()
        if any(k in q for k in ('lab', 'blood', 'hemoglobin', 'cbc', 'liver',
                                  'kidney', 'glucose', 'creatinine', 'thyroid',
                                  'platelet', 'wbc', 'rbc', 'anemia')):
            return 'lab'
        if any(k in q for k in ('x-ray', 'xray', 'ct ', 'ct,', 'mri',
                                  'ultrasound', 'imaging', 'scan', 'radiol',
                                  'chest', 'infiltrate', 'pneumonia')):
            return 'radiology'
        if any(k in q for k in ('allerg', 'allergic', 'reaction',
                                  'anaphylaxis', 'cross-react', 'peanut',
                                  'penicillin', 'latex')):
            return 'allergy'
        return 'general'

    def check_permission(self, role: ClinicalRole, agent_type: str) -> bool:
        return agent_type in self.role_permissions.get(role, []) or agent_type == 'general'

    def process_with_tracing(self, query: str, role: ClinicalRole) -> Dict:
        trace = langfuse.trace(
            name=f"orchestration-{role.value}",
            session_id=SESSION_ID,
            input={"query": query, "role": role.value}
        ) if langfuse else None

        # Route
        routing_span = trace.span(name="routing") if trace else None
        agent_type = self.route_query(query)
        if routing_span:
            routing_span.end(output={"agent_type": agent_type})

        # Permission check
        perm_span = trace.span(name="permission-check") if trace else None
        allowed = self.check_permission(role, agent_type)
        if perm_span:
            perm_span.end(output={"allowed": allowed, "agent_type": agent_type})

        if not allowed:
            allowed_types = self.role_permissions.get(role, [])
            if trace:
                trace.update(output={"error": "access_denied"})
            return {
                "error": "Access denied",
                "role": role.value,
                "requested": agent_type,
                "allowed_types": allowed_types
            }

        # RAG retrieval
        rag_span = trace.span(name="rag-retrieval") if trace else None
        context_results = self.vector_store.search_with_tracing(
            "healthcare_all", query, n_results=3
        )
        context_docs = context_results['documents'][0] if context_results['documents'] else []
        context_text = '\n\n'.join(context_docs) if context_docs else ""
        if rag_span:
            rag_span.end(output={"docs_retrieved": len(context_docs)})

        # Agent execution
        enriched_query = f"{query}\n\nContext from clinical records:\n{context_text}" if context_text else query
        agent_span = trace.span(name=f"agent-{agent_type}") if trace else None

        if agent_type == 'lab':
            result = self.lab_agent.interpret(enriched_query)
        elif agent_type == 'radiology':
            result = self.radiology_agent.analyze(enriched_query)
        elif agent_type == 'allergy':
            result = self.allergy_agent.assess(enriched_query)
        else:
            result = {"success": False, "error": f"No agent for type: {agent_type}"}

        if agent_span:
            agent_span.end(output={"success": result.get("success")})
        if trace:
            trace.update(output={"agent_type": agent_type, "success": result.get("success")})

        return {
            'query':      query,
            'role':       role.value,
            'agent_type': agent_type,
            'result':     result,
            'context_docs_used': len(context_docs)
        }


orchestrator = TracedClinicalOrchestrator()
print("TracedClinicalOrchestrator initialized")

In [ ]:
# ============================================================================
# VALIDATION CELL 5.1: Test Orchestration & RBAC
# ============================================================================

print("ORCHESTRATION & RBAC TESTS")
print("="*60)

test_cases = [
    # (query, role, expected_routing, should_be_allowed)
    ("What do my blood test results mean?", ClinicalRole.PATIENT, "lab", False),
    ("Interpret the CBC report for anemia", ClinicalRole.CLINICIAN, "lab", True),
    ("Analyze the chest X-ray findings", ClinicalRole.RADIOLOGIST, "radiology", True),
    ("Check peanut allergy cross-reactions", ClinicalRole.PATIENT, "allergy", True),
    ("Full patient workup including labs and imaging", ClinicalRole.COORDINATOR, "lab", True),
]

for i, (query, role, expected_routing, should_allow) in enumerate(test_cases, 1):
    print(f"\n[Test {i}] {role.value.upper()}")
    print(f"Query: '{query[:50]}...'")

    result = orchestrator.process_with_tracing(query, role)

    actual_routing = orchestrator.route_query(query)
    actual_allowed = orchestrator.check_permission(role, actual_routing)

    print(f"  Routing: {actual_routing} (expected: {expected_routing})")
    print(f"  Allowed: {actual_allowed} (expected: {should_allow})")

    if actual_routing == expected_routing and actual_allowed == should_allow:
        print("  PASS")
    else:
        print("  NEEDS WORK")

---
# WEEK 6: Medical Knowledge Graph
---

## Learning Objectives
- Build a medical knowledge graph using NetworkX
- Model relationships: symptoms -> diagnoses, tests -> diagnoses
- Implement graph traversal for clinical reasoning

## Difficulty: Advanced

**HEALTHCARE CHALLENGE - Clinical Reasoning**:
Knowledge graphs enable:
- Symptom-to-diagnosis reasoning
- Allergen cross-reactivity mapping
- Treatment recommendations based on diagnosis
- Contraindication checking

In [ ]:
# ============================================================================
# TODO 6.1: BUILD MEDICAL KNOWLEDGE GRAPH
# ============================================================================

import networkx as nx
import matplotlib.pyplot as plt

class MedicalKnowledgeGraph:
    """
    Knowledge graph for medical reasoning.
    """

    def __init__(self):
        self.graph = nx.DiGraph()
        self._build_medical_ontology()

    def _build_medical_ontology(self):
        """Build the medical knowledge graph with clinical relationships."""

        # ── Lab Tests → Diagnoses (CONFIRMS) ──────────────────────────────
        lab_tests = {
            'CBC':           ['anemia', 'infection', 'leukemia', 'thrombocytopenia'],
            'LFT':           ['hepatitis', 'cirrhosis', 'fatty_liver'],
            'KFT':           ['kidney_disease', 'dehydration'],
            'Thyroid_Panel': ['hypothyroidism', 'hyperthyroidism'],
            'Blood_Glucose': ['diabetes', 'hypoglycemia'],
            'Lipid_Panel':   ['hyperlipidemia', 'cardiovascular_disease'],
        }
        for test, diagnoses in lab_tests.items():
            self.graph.add_node(test, type='TEST')
            for d in diagnoses:
                self.graph.add_node(d, type='DIAGNOSIS')
                self.graph.add_edge(test, d, relation='CONFIRMS')

        # ── Symptoms → Diagnoses (INDICATES) ──────────────────────────────
        symptom_map = {
            'fatigue':              ['anemia', 'hypothyroidism', 'infection', 'diabetes'],
            'fever':                ['infection', 'inflammation', 'leukemia'],
            'jaundice':             ['hepatitis', 'cirrhosis'],
            'shortness_of_breath':  ['anemia', 'pneumonia', 'heart_failure'],
            'weight_gain':          ['hypothyroidism', 'diabetes'],
            'weight_loss':          ['hyperthyroidism', 'diabetes', 'infection'],
            'frequent_urination':   ['diabetes', 'kidney_disease'],
            'swelling':             ['kidney_disease', 'heart_failure', 'cirrhosis'],
            'chest_pain':           ['heart_failure', 'cardiovascular_disease', 'pneumonia'],
            'palpitations':         ['hyperthyroidism', 'cardiovascular_disease'],
        }
        for symptom, diagnoses in symptom_map.items():
            self.graph.add_node(symptom, type='SYMPTOM')
            for d in diagnoses:
                self.graph.add_node(d, type='DIAGNOSIS')
                self.graph.add_edge(symptom, d, relation='INDICATES')

        # ── Allergen Cross-Reactivity (CROSS_REACTIVE) ────────────────────
        cross_reactivity = {
            'peanut':     ['tree_nuts', 'legumes', 'lupin', 'soy'],
            'penicillin': ['amoxicillin', 'ampicillin', 'cephalosporins'],
            'aspirin':    ['nsaids', 'ibuprofen', 'naproxen'],
            'latex':      ['banana', 'avocado', 'kiwi', 'chestnut'],
            'shellfish':  ['shrimp', 'crab', 'lobster'],
            'wheat':      ['gluten', 'barley', 'rye'],
        }
        for allergen, related in cross_reactivity.items():
            self.graph.add_node(allergen, type='ALLERGEN')
            for r in related:
                self.graph.add_node(r, type='ALLERGEN')
                self.graph.add_edge(allergen, r, relation='CROSS_REACTIVE')

        # ── Treatments → Diagnoses (TREATS) ───────────────────────────────
        treatment_map = {
            'iron_supplements':  ['anemia'],
            'antibiotics':       ['infection', 'pneumonia'],
            'levothyroxine':     ['hypothyroidism'],
            'antithyroid_drugs': ['hyperthyroidism'],
            'insulin':           ['diabetes'],
            'diuretics':         ['heart_failure', 'kidney_disease'],
            'antivirals':        ['hepatitis'],
            'statins':           ['hyperlipidemia', 'cardiovascular_disease'],
            'chemotherapy':      ['leukemia'],
            'dialysis':          ['kidney_disease'],
        }
        for treatment, diagnoses in treatment_map.items():
            self.graph.add_node(treatment, type='TREATMENT')
            for d in diagnoses:
                self.graph.add_node(d, type='DIAGNOSIS')
                self.graph.add_edge(treatment, d, relation='TREATS')

    def find_diagnoses_for_symptoms(self, symptoms: List[str]) -> Dict:
        """Find possible diagnoses based on symptoms, ranked by matching count."""
        diagnosis_scores = {}

        for symptom in symptoms:
            if symptom in self.graph:
                for _, target, data in self.graph.out_edges(symptom, data=True):
                    if data.get('relation') == 'INDICATES':
                        diagnosis_scores[target] = diagnosis_scores.get(target, 0) + 1

        sorted_diagnoses = sorted(diagnosis_scores.items(), key=lambda x: -x[1])
        return {'symptoms': symptoms, 'diagnoses': sorted_diagnoses}

    def get_cross_reactive(self, allergen: str) -> List[str]:
        """Get all cross-reactive allergens."""
        if allergen not in self.graph:
            return []
        return [
            target
            for _, target, data in self.graph.out_edges(allergen, data=True)
            if data.get('relation') == 'CROSS_REACTIVE'
        ]

    def get_treatments_for_diagnosis(self, diagnosis: str) -> List[str]:
        """Get treatments for a diagnosis."""
        if diagnosis not in self.graph:
            return []
        return [
            source
            for source, _, data in self.graph.in_edges(diagnosis, data=True)
            if data.get('relation') == 'TREATS'
        ]


# Initialize knowledge graph
medical_kg = MedicalKnowledgeGraph()

print(f"Medical Knowledge Graph created:")
print(f"  Nodes: {medical_kg.graph.number_of_nodes()}")
print(f"  Edges: {medical_kg.graph.number_of_edges()}")

In [ ]:
# ============================================================================
# TODO 6.2: VISUALIZE THE KNOWLEDGE GRAPH
# ============================================================================

print("KNOWLEDGE GRAPH VISUALIZATION")
print("="*60)

# Color mapping for node types
color_map = {
    'SYMPTOM': '#ff6b6b',      # Red
    'DIAGNOSIS': '#4ecdc4',    # Teal
    'TEST': '#45b7d1',         # Blue
    'TREATMENT': '#96ceb4',    # Green
    'ALLERGEN': '#ffeaa7'      # Yellow
}

# Get node colors
node_colors = [color_map.get(medical_kg.graph.nodes[n].get('type', 'OTHER'), '#95a5a6')
               for n in medical_kg.graph.nodes()]

# Create visualization
plt.figure(figsize=(14, 10))
pos = nx.spring_layout(medical_kg.graph, k=2, iterations=50, seed=42)

nx.draw_networkx_nodes(medical_kg.graph, pos, node_color=node_colors,
                       node_size=800, alpha=0.9)
nx.draw_networkx_edges(medical_kg.graph, pos, edge_color='gray',
                       arrows=True, arrowsize=15, alpha=0.6)
nx.draw_networkx_labels(medical_kg.graph, pos, font_size=8)

# Legend
legend_elements = [plt.scatter([], [], c=color, s=100, label=node_type)
                   for node_type, color in color_map.items()]
plt.legend(handles=legend_elements, loc='upper left', title='Node Types')

plt.title('Medical Knowledge Graph', fontsize=14)
plt.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================================
# VALIDATION CELL 6.1: Test Knowledge Graph Reasoning
# ============================================================================

print("KNOWLEDGE GRAPH REASONING TESTS")
print("="*60)

# Test 1: Symptom to Diagnosis
print("\n[Test 1] Symptom-to-Diagnosis Reasoning")
symptoms = ['fatigue', 'shortness_of_breath']
result = medical_kg.find_diagnoses_for_symptoms(symptoms)
print(f"Symptoms: {symptoms}")
print(f"Possible diagnoses: {result['diagnoses'][:5]}")

# Test 2: Cross-Reactivity
print("\n[Test 2] Allergen Cross-Reactivity")
for allergen in ['peanut', 'penicillin', 'latex']:
    cross = medical_kg.get_cross_reactive(allergen)
    print(f"  {allergen} -> {cross}")

# Test 3: Treatment Lookup
print("\n[Test 3] Treatment Recommendations")
for diagnosis in ['anemia', 'infection', 'hypothyroidism']:
    treatments = medical_kg.get_treatments_for_diagnosis(diagnosis)
    print(f"  {diagnosis} -> {treatments}")

# Validation
print("\n" + "-"*40)
if medical_kg.graph.number_of_nodes() >= 20:
    print("VALIDATION PASSED: Graph has sufficient nodes")
else:
    print("NEEDS WORK: Add more nodes to the knowledge graph")